# 第96章 概率校准与Brier Score

检查分类概率是否可信，并使用校准曲线、Brier Score 和 CalibratedClassifierCV 改善概率。


## 先解决一个小问题

围绕“概率校准与Brier Score”完成一个可验证的小型建模实验：先明确输入和目标，再比较方法带来的变化。检查分类概率是否可信，并使用校准曲线、Brier Score 和 CalibratedClassifierCV 改善概率。


## 这章为什么先学

这是“机器学习”建模主线中的第 96 章，重点放在“概率校准与Brier Score”对应的一个具体决策，而不是重复完整流程。


## 开始前确认

- 能够使用 pandas 读取、筛选和汇总数据
- 理解训练集、测试集和基本统计指标
- 本章会进一步练习：区分排序与校准、计算 Brier Score、读取校准分箱


## 做完要留下什么

完成一份围绕“概率校准与Brier Score”的可运行实验：包含数据准备、方法执行、指标或图表证据，以及一句有边界的结论。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 区分排序与校准
- 计算 Brier Score
- 读取校准分箱
- 使用交叉验证校准模型


## 核心概念

- Brier=\(n^{-1}\sum_i(p_i-y_i)^2\)
- 校准概率 0.7 应约有 70% 正例
- sigmoid 稳健、isotonic 更灵活
- 校准必须使用独立数据或内部交叉验证


## 示例 1：数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True); X, y=data.data, data.target
X_train, X_test, y_train, y_test=train_test_split(X, y, stratify=y, random_state=96)


## 示例 2：模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd

base = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=96)
raw = base.fit(X_train, y_train); calibrated=CalibratedClassifierCV(base, method='sigmoid', cv=5).fit(X_train, y_train)
rows = []
for name, m in [('raw', raw), ('calibrated', calibrated)]:
    p = m.predict_proba(X_test)[:,1]; rows.append([name, brier_score_loss(y_test, p), roc_auc_score(y_test, p)])
display(pd.DataFrame(rows, columns=['model', 'Brier', 'ROC_AUC']).set_index('model').round(4))
obs, forecast=calibration_curve(y_test, calibrated.predict_proba(X_test)[:,1], n_bins=6)
display(pd.DataFrame({'预测概率':forecast, '实际比例':obs}).round(3))


## 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 常见误区

- AUC 高就认为概率可信
- 在测试集上拟合校准器
- 小样本使用过细的校准分箱
- 部署后不监控概率漂移


## 综合练习

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
practice_brier = {name:brier_score_loss(y_test, m.predict_proba(X_test)[:,1]) for name, m in [('raw', raw), ('calibrated', calibrated)]}
print(practice_brier)

# 自检
assert all(0<=v<=1 for v in practice_brier.values())


## 本章小结

检查分类概率是否可信，并使用校准曲线、Brier Score 和 CalibratedClassifierCV 改善概率。


### 你已经掌握

- 区分排序与校准
- 计算 Brier Score
- 读取校准分箱
- 使用交叉验证校准模型


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 数据与问题定义 | 先明确样本、特征、目标和验证方式，再训练模型。 | 参见本节示例 |
| 模型、公式与诊断 | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | `base.fit()`、`m.predict_proba()`、`rows.append()`、`pd.DataFrame()` |


### 需要注意

- AUC 高就认为概率可信
- 在测试集上拟合校准器
- 小样本使用过细的校准分箱
- 部署后不监控概率漂移


### 完成检查

- [ ] 能够区分排序与校准
- [ ] 能够计算 Brier Score
- [ ] 能够读取校准分箱
- [ ] 能够使用交叉验证校准模型


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
